## Full RAG Pipeline

End-to-end Retrieval-Augmented Generation (RAG) pipeline for legal document Q&A.

> **About RAG:** RAG combines information retrieval (finding relevant documents) with text generation (creating answers). This pipeline retrieves relevant legal document chunks from ChromaDB and uses Groq LLM to generate accurate, cited answers.

### RAG Architecture

```
User Question
     ↓
Generate Query Embedding
     ↓
Retrieve from ChromaDB (top-k similar chunks)
     ↓
Build Context (with truncation if needed)
     ↓
Format Prompt (CONSTRAINED_RAG_PROMPT)
     ↓
Call Groq LLM
     ↓
Return Answer + Citations
```

---

## Phase 1: Setup (Run Once)

### Import Libraries

In [1]:
# Core libraries
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter
from llama_index.llms.groq import Groq
from llama_index.core import Settings

# Vector database and embeddings
import chromadb
from sentence_transformers import SentenceTransformer

# Environment and utilities
from dotenv import load_dotenv
import os
import time
import re
import json
from pathlib import Path
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from openai import APIStatusError

### Configuration

In [2]:
# Global constants
MODEL_NAME = "all-MiniLM-L6-v2"  # Embedding model
GROQ_MODEL = "llama-3.1-8b-instant"  # LLM model (best for free tier)
CONTEXT_WINDOW = 131072  # 128K context support
CHUNK_SIZE = 512
CHUNK_OVERLAP = 50
MAX_CONTEXT_LENGTH = 15000  # Characters to stay within 6K TPM limit
TOP_K_RESULTS = 3  # Number of chunks to retrieve

# Paths
BASE_DIR = Path("../data")
TXT_DIR = BASE_DIR / "txt"

# ChromaDB collection name
COLLECTION_NAME = "legal-docs"

### Load Environment Variables

In [3]:
# Load API keys from .env file
load_dotenv()

# Get Groq API key
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY or GROQ_API_KEY == "your_groq_api_key_here":
    print("⚠ Warning: GROQ_API_KEY not set!")
    print("Please create a .env file with your API key.")
    print("See .env.example for template.")
else:
    print("✓ GROQ_API_KEY loaded successfully")
    print(f"  Key starts with: {GROQ_API_KEY[:8]}...")

✓ GROQ_API_KEY loaded successfully
  Key starts with: gsk_Q5Ti...


### Initialize Components

In [4]:
# Initialize embedding model (load once, reuse everywhere)
print(f"Loading embedding model: {MODEL_NAME}...")
model = SentenceTransformer(MODEL_NAME)
print("✓ Embedding model loaded")

# Initialize Groq LLM
print(f"Initializing Groq LLM: {GROQ_MODEL}...")
llm = Groq(
    model=GROQ_MODEL,
    api_key=GROQ_API_KEY,
    context_window=CONTEXT_WINDOW
)
Settings.llm = llm
print("✓ Groq LLM initialized")

# Connect to ChromaDB (running in Docker)
print("Connecting to ChromaDB (localhost:8000)...")
chroma_client = chromadb.HttpClient(host="localhost", port="8000")
print(f"✓ ChromaDB connected: {chroma_client.heartbeat()}")

# Get or create collection
collection = chroma_client.get_or_create_collection(name=COLLECTION_NAME)
print(f"✓ Collection ready: {collection.name}")

Loading embedding model: all-MiniLM-L6-v2...
✓ Embedding model loaded
Initializing Groq LLM: llama-3.1-8b-instant...
✓ Groq LLM initialized
Connecting to ChromaDB (localhost:8000)...
✓ ChromaDB connected: 1773819648668397507
✓ Collection ready: legal-docs


---

## Phase 2: Document Processing (Run Once to Populate ChromaDB)

### Helper Functions

In [5]:
def estimate_tokens(text):
    """
    Rough estimate of tokens in text (4 chars ≈ 1 token for English).
    
    Args:
        text: Input text string
    
    Returns:
        Estimated token count
    """
    return len(text) // 4


def limit_context(context, max_length=MAX_CONTEXT_LENGTH, verbose=True):
    """
    Limit context length to avoid Groq API token rate limits.
    
    Args:
        context: The full context string from retrieved documents
        max_length: Maximum character limit (default: 15,000 chars ≈ 3,750 tokens)
        verbose: Whether to print warning when truncating
    
    Returns:
        Truncated context string with notification if truncated
    
    Note:
        Groq free tier has 6,000 TPM (tokens per minute) limit for llama-3.1-8b-instant.
        Keep context under 4,000 tokens to leave room for prompt + response.
    """
    if len(context) > max_length:
        if verbose:
            est_before = estimate_tokens(context)
            est_after = estimate_tokens(context[:max_length])
            print(f"⚠ Context truncated from {len(context):,} to {max_length:,} chars")
            print(f"  (est. tokens: {est_before:,} → {est_after:,} to fit 6K TPM limit)")
        return context[:max_length] + "\n\n[Context truncated to fit token limits]"
    return context


def build_context_from_results(results, max_length=MAX_CONTEXT_LENGTH):
    """
    Build context from ChromaDB query results with automatic length limiting.
    
    Args:
        results: ChromaDB query results object
        max_length: Maximum character limit for final context
    
    Returns:
        Tuple of (context_string, num_docs)
    """
    context_docs = results["documents"][0]
    context = "\n\n".join(context_docs)
    context = limit_context(context, max_length)
    return context, len(context_docs)


# Metadata extraction functions (from Notebook 03)
def extract_insc_citation(text):
    """Extract Supreme Court case citations (e.g., '2025 INSC 443')."""
    pattern = r'(\d{4})\s+INSC\s+(\d+)'
    return re.findall(pattern, text)


def extract_scc_citation(text):
    """Extract SCC citations (e.g., '(2006) 1 SCC 1')."""
    pattern = r'\((\d{4})\)\s+(\d+)\s+SCC\s+(\d+)'
    return re.findall(pattern, text)


def extract_dates(text):
    """Extract dates (e.g., '24 March, 2025')."""
    pattern = r'(\d{1,2}\s+\w+\s+\d{4})'
    return re.findall(pattern, text)[:5]  # Limit to first 5 dates


def extract_bench(text):
    """Extract court bench names."""
    pattern = r'Bench:\s*([A-Z][^.\n]+)'
    return re.findall(pattern, text)

### Load Documents

In [6]:
# Load extracted txt files from data/txt/
print(f"Loading documents from {TXT_DIR}...")

try:
    documents = SimpleDirectoryReader(str(TXT_DIR)).load_data()
    print(f"✓ Loaded {len(documents)} documents")
except FileNotFoundError:
    print(f"⚠ TXT directory not found: {TXT_DIR}")
    print("Run Notebook 01 first to extract text files from PDFs/DOCXs")
    documents = []
except Exception as e:
    print(f"⚠ Error loading documents: {e}")
    documents = []

Loading documents from ../data/txt...
✓ Loaded 4 documents


### Chunk & Enrich with Metadata

In [7]:
# Create sentence splitter
splitter = SentenceSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)
print(f"✓ Splitter created: chunk_size={CHUNK_SIZE}, chunk_overlap={CHUNK_OVERLAP}")

# Split documents into chunks
if documents:
    nodes = splitter.get_nodes_from_documents(documents)
    print(f"✓ Created {len(nodes)} chunks from {len(documents)} documents")
else:
    nodes = []
    print("⚠ No documents to chunk")

✓ Splitter created: chunk_size=512, chunk_overlap=50
✓ Created 57 chunks from 4 documents


In [8]:
# Add extracted metadata to each chunk
if nodes:
    for node in nodes:
        text = node.text
        
        # Extract metadata
        insc_citations = extract_insc_citation(text)
        scc_citations = extract_scc_citation(text)
        dates = extract_dates(text)
        bench = extract_bench(text)
        
        # Add to node metadata (only if found)
        if insc_citations:
            node.metadata["insc_citations"] = str(insc_citations)
        if scc_citations:
            node.metadata["scc_citations"] = str(scc_citations)
        if dates:
            node.metadata["dates"] = str(dates)
        if bench:
            node.metadata["bench"] = str(bench)
    
    # Count chunks with metadata
    metadata_counts = {
        "insc_citations": sum(1 for n in nodes if "insc_citations" in n.metadata),
        "scc_citations": sum(1 for n in nodes if "scc_citations" in n.metadata),
        "dates": sum(1 for n in nodes if "dates" in n.metadata),
        "bench": sum(1 for n in nodes if "bench" in n.metadata)
    }
    
    print(f"✓ Added metadata to {len(nodes)} chunks")
    print(f"  Chunks with INSC citations: {metadata_counts['insc_citations']}")
    print(f"  Chunks with SCC citations: {metadata_counts['scc_citations']}")
    print(f"  Chunks with dates: {metadata_counts['dates']}")
    print(f"  Chunks with bench info: {metadata_counts['bench']}")

✓ Added metadata to 57 chunks
  Chunks with INSC citations: 4
  Chunks with SCC citations: 8
  Chunks with dates: 37
  Chunks with bench info: 4


### Embed & Store in ChromaDB

In [9]:
# Prepare data for ChromaDB
if nodes:
    docs = []
    metadatas = []
    ids = []
    
    for i, node in enumerate(nodes):
        docs.append(node.text)
        metadatas.append(node.metadata)
        ids.append(f"chunk_{i}")
    
    print(f"✓ Prepared {len(docs)} chunks for insertion")
    
    # Generate embeddings
    print(f"Generating embeddings for {len(docs)} chunks...")
    embeddings = model.encode(docs).tolist()
    print(f"✓ Generated {len(embeddings)} embeddings ({len(embeddings[0])} dimensions)")
    
    # Clear existing data and add to ChromaDB
    print(f"Adding documents to ChromaDB collection '{COLLECTION_NAME}'...")
    chroma_client.delete_collection(COLLECTION_NAME)
    collection = chroma_client.get_or_create_collection(name=COLLECTION_NAME)
    
    collection.add(
        documents=docs,
        embeddings=embeddings,
        metadatas=metadatas,
        ids=ids
    )
    
    print(f"✓ Added {len(ids)} chunks to ChromaDB")
else:
    print("⚠ No chunks to embed and store")

✓ Prepared 57 chunks for insertion
Generating embeddings for 57 chunks...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

2026-03-18 13:11:20,502 - INFO - HTTP Request: DELETE http://localhost:8000/api/v2/tenants/default_tenant/databases/default_database/collections/legal-docs "HTTP/1.1 200 OK"
2026-03-18 13:11:20,513 - INFO - HTTP Request: POST http://localhost:8000/api/v2/tenants/default_tenant/databases/default_database/collections "HTTP/1.1 200 OK"
2026-03-18 13:11:20,521 - INFO - HTTP Request: GET http://localhost:8000/api/v2/pre-flight-checks "HTTP/1.1 200 OK"


✓ Generated 57 embeddings (384 dimensions)
Adding documents to ChromaDB collection 'legal-docs'...


2026-03-18 13:11:20,811 - INFO - HTTP Request: POST http://localhost:8000/api/v2/tenants/default_tenant/databases/default_database/collections/69f23a87-3fd8-4a7a-b55e-6dc9e92dc714/add "HTTP/1.1 201 Created"


✓ Added 57 chunks to ChromaDB


---

## Phase 3: Interactive Q&A (Repeatable)

### RAG Prompt Template

In [10]:
# Constrained RAG prompt for accuracy (from Notebook 07)
CONSTRAINED_RAG_PROMPT = """
You are a precise legal assistant. Answer questions using ONLY the provided context.

Context:
{context}

Question: {question}

Instructions:
1. Answer ONLY based on the provided context - do not use external knowledge
2. Cite specific sections, paragraphs, or sources when available
3. If the answer is not in the context, state "I don't have enough information in the provided context"
4. Do not make assumptions or inferences beyond what is explicitly stated
5. Keep responses concise but complete

Answer:
"""

### Retry Logic for Groq API

In [11]:
def is_rate_limit_error(exception):
    """Check if exception is a rate limit error (429 or 413)."""
    if isinstance(exception, APIStatusError):
        return exception.status_code in [429, 413]
    return False


@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=5, max=60),
    retry=retry_if_exception_type(APIStatusError),
    reraise=True
)
def llm_complete_with_retry(llm, prompt):
    """
    Call LLM completion with retry logic for rate limit errors.
    
    Args:
        llm: Groq LLM instance
        prompt: Input prompt
    
    Returns:
        LLM completion response
    
    Note:
        Retries on 429 (rate limit) and 413 (request too large) errors
        with exponential backoff: 5s, 10s, 20s, etc.
    """
    try:
        return llm.complete(prompt)
    except APIStatusError as e:
        if is_rate_limit_error(e):
            print(f"⚠ Rate limit hit: {e}")
            print(f"  Retrying with exponential backoff...")
        raise

### Main Q&A Function

In [12]:
def ask_question(query, top_k=TOP_K_RESULTS, verbose=True):
    """
    Ask a question and get an answer from the RAG pipeline.
    
    This is the main function that can be imported into a Streamlit app.
    
    Args:
        query: User's question string
        top_k: Number of chunks to retrieve from ChromaDB (default: 3)
        verbose: Whether to print intermediate steps
    
    Returns:
        dict with keys:
            - 'answer': The LLM-generated answer
            - 'sources': List of source document names
            - 'response_time': Time taken in seconds
            - 'chunks_used': Number of chunks retrieved
    
    Example:
        >>> result = ask_question("What are the forest conservation issues?")
        >>> print(result['answer'])
        >>> print(result['sources'])
    """
    start_time = time.time()
    
    # Step 1: Generate query embedding
    if verbose:
        print(f"\n🔍 Processing question: {query}")
        print("  Step 1: Generating query embedding...")
    query_embedding = model.encode([query]).tolist()
    
    # Step 2: Retrieve from ChromaDB
    if verbose:
        print(f"  Step 2: Retrieving top {top_k} chunks from ChromaDB...")
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )
    
    # Check if we got results
    if not results["documents"][0]:
        return {
            "answer": "I don't have enough information in the provided context to answer this question.",
            "sources": [],
            "response_time": time.time() - start_time,
            "chunks_used": 0
        }
    
    # Step 3: Build context
    if verbose:
        print("  Step 3: Building context...")
    context, num_docs = build_context_from_results(results, MAX_CONTEXT_LENGTH)
    
    # Extract source document names
    sources = list(set(
        meta.get("file_name", "unknown") 
        for meta in results["metadatas"][0]
    ))
    
    # Step 4: Format prompt
    if verbose:
        print("  Step 4: Formatting prompt...")
    formatted_prompt = CONSTRAINED_RAG_PROMPT.format(
        context=context,
        question=query
    )
    
    # Estimate tokens
    est_tokens = estimate_tokens(formatted_prompt)
    if verbose:
        print(f"  Estimated prompt size: {est_tokens:,} tokens")
    
    # Step 5: Call Groq LLM
    if verbose:
        print("  Step 5: Calling Groq LLM...")
    response = llm_complete_with_retry(llm, formatted_prompt)
    
    # Calculate response time
    response_time = time.time() - start_time
    
    if verbose:
        print(f"✓ Answer generated in {response_time:.2f} seconds")
    
    return {
        "answer": str(response),
        "sources": sources,
        "response_time": response_time,
        "chunks_used": num_docs
    }

### Example Query

In [13]:
# Test the pipeline with a sample question
test_query = "What are the forest conservation issues?"
result = ask_question(test_query, verbose=True)

print("\n" + "=" * 60)
print(f"Question: {test_query}")
print("=" * 60)
print(f"\nAnswer:\n{result['answer']}")
print(f"\nSources:")
for source in result['sources']:
    print(f"  • {source}")
print(f"\n⏱ Response time: {result['response_time']:.2f} seconds")
print(f"📄 Chunks used: {result['chunks_used']}")


🔍 Processing question: What are the forest conservation issues?
  Step 1: Generating query embedding...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-03-18 13:11:51,963 - INFO - HTTP Request: POST http://localhost:8000/api/v2/tenants/default_tenant/databases/default_database/collections/69f23a87-3fd8-4a7a-b55e-6dc9e92dc714/query "HTTP/1.1 200 OK"


  Step 2: Retrieving top 3 chunks from ChromaDB...
  Step 3: Building context...
  Step 4: Formatting prompt...
  Estimated prompt size: 1,336 tokens
  Step 5: Calling Groq LLM...


2026-03-18 13:11:55,506 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


✓ Answer generated in 3.67 seconds

Question: What are the forest conservation issues?

Answer:
Based on the provided context, the forest conservation issues mentioned are:

1. Destruction of forests due to agriculture (Paragraph 1).
2. Depletion of forests due to rapid urbanization, unchecked industrialization, encroachments, etc. (Paragraph 3).
3. Inadequate forest cover in India, which is about 21.76% of the total landmass of the country (Paragraph 4).
4. Encroachments on forest areas, with almost 13,000 sq. km of forests under encroachment (Paragraph 5).
5. Need to transform from an anthropocentric approach to an ecocentric approach to protect the environment (Paragraph 7).

Additionally, the context mentions the following specific issues:

- The Singampatti Zamin forest lands in Tamil Nadu have been exploited for over 95 years, with the lease holders clearing out the forests and cultivating crops (Paragraph 8).

Sources:
  • A_John_Kennedy_vs_The_State_Of_Tamil_Nadu_on_24_March_20

### Interactive Q&A Session

In [14]:
# Interactive loop - mimics app behavior
# Run this cell to start a Q&A session

print("\n" + "=" * 60)
print("🤖 Lexi Bot - Legal Document Q&A")
print("=" * 60)
print(f"\n✓ ChromaDB collection: {collection.name} ({collection.count()} chunks)")
print(f"✓ Groq LLM: {GROQ_MODEL}")
print(f"✓ Embedding model: {MODEL_NAME}")
print("\nAsk legal questions based on the loaded documents.")
print("Type 'quit' or 'exit' to end the session.\n")

while True:
    try:
        # Get user input
        query = input("\n📝 Ask a legal question (or 'quit'): ").strip()
        
        # Check for exit command
        if query.lower() in ['quit', 'exit', 'q']:
            print("\n✓ Session ended. Goodbye!")
            break
        
        # Skip empty input
        if not query:
            print("⚠ Please enter a question.")
            continue
        
        # Get answer from RAG pipeline
        result = ask_question(query, verbose=False)
        
        # Display answer
        print("\n" + "-" * 60)
        print(f"💡 Answer:\n{result['answer']}")
        print("\n📚 Sources:")
        if result['sources']:
            for source in result['sources']:
                print(f"  • {source}")
        else:
            print("  (No sources - answer not found in context)")
        print(f"\n⏱ Response time: {result['response_time']:.2f}s | Chunks: {result['chunks_used']}")
        print("-" * 60)
        
    except KeyboardInterrupt:
        print("\n\n⚠ Session interrupted. Goodbye!")
        break
    except Exception as e:
        print(f"\n⚠ Error: {e}")
        print("Please try again.")

2026-03-18 13:12:22,648 - INFO - HTTP Request: GET http://localhost:8000/api/v2/tenants/default_tenant/databases/default_database/collections/69f23a87-3fd8-4a7a-b55e-6dc9e92dc714/count?read_level=index_and_wal "HTTP/1.1 200 OK"



🤖 Lexi Bot - Legal Document Q&A

✓ ChromaDB collection: legal-docs (57 chunks)
✓ Groq LLM: llama-3.1-8b-instant
✓ Embedding model: all-MiniLM-L6-v2

Ask legal questions based on the loaded documents.
Type 'quit' or 'exit' to end the session.



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-03-18 13:12:30,698 - INFO - HTTP Request: POST http://localhost:8000/api/v2/tenants/default_tenant/databases/default_database/collections/69f23a87-3fd8-4a7a-b55e-6dc9e92dc714/query "HTTP/1.1 200 OK"
2026-03-18 13:12:31,949 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"



------------------------------------------------------------
💡 Answer:
Based on the provided context, the case revolves around two appeals (CIVIL APPEAL NO(S). 11070 – 11071 OF 2024) filed by the appellant. The appeals were dismissed by the NCLAT, Chennai, as they were deemed to be beyond the period of limitation.

The appellant claimed that the appeals were filed within the permissible period, citing Section 12(3) of the Limitation Act and Section 61 of the IBC, specifically the proviso to sub-section (2) which allows an additional 15 days to file an appeal beyond the initial 30 days.

However, the NCLAT found that the appellant was guilty of suppression of correct facts and making wrong averments in the grounds of appeal, including mis-stating the fact that certified copies were applied for. The NCLAT also dismissed the applications for condonation of delay, leading to the dismissal of the appeals.

The learned senior counsel for the appellant argued that the appeals should have bee

---

## Summary & Next Steps

In [16]:
# Pipeline statistics
print("\n" + "=" * 60)
print("📊 Lexi Bot RAG Pipeline Summary")
print("=" * 60)

print(f"\n✅ Components Initialized:")
print(f"  • Embedding model: {MODEL_NAME}")
print(f"  • Groq LLM: {GROQ_MODEL}")
print(f"  • ChromaDB collection: {COLLECTION_NAME}")

print(f"\n📁 Document Processing:")
print(f"  • Documents loaded: {len(documents)}")
print(f"  • Chunks created: {len(nodes)}")
print(f"  • Chunks in ChromaDB: {collection.count()}")

print(f"\n⚙️ Configuration:")
print(f"  • Chunk size: {CHUNK_SIZE} tokens")
print(f"  • Chunk overlap: {CHUNK_OVERLAP} tokens")
print(f"  • Max context length: {MAX_CONTEXT_LENGTH:,} chars")
print(f"  • Top-K retrieval: {TOP_K_RESULTS} chunks")

print(f"\n🎯 Ready for Q&A!")
print(f"  • Use ask_question(query) function")
print(f"  • Or run the interactive loop above")

2026-03-18 13:13:25,103 - INFO - HTTP Request: GET http://localhost:8000/api/v2/tenants/default_tenant/databases/default_database/collections/69f23a87-3fd8-4a7a-b55e-6dc9e92dc714/count?read_level=index_and_wal "HTTP/1.1 200 OK"



📊 Lexi Bot RAG Pipeline Summary

✅ Components Initialized:
  • Embedding model: all-MiniLM-L6-v2
  • Groq LLM: llama-3.1-8b-instant
  • ChromaDB collection: legal-docs

📁 Document Processing:
  • Documents loaded: 4
  • Chunks created: 57
  • Chunks in ChromaDB: 57

⚙️ Configuration:
  • Chunk size: 512 tokens
  • Chunk overlap: 50 tokens
  • Max context length: 15,000 chars
  • Top-K retrieval: 3 chunks

🎯 Ready for Q&A!
  • Use ask_question(query) function
  • Or run the interactive loop above
